<a href="https://colab.research.google.com/github/sarah2005-cyber/auspex/blob/main/Universal_Audio_Steganalysis_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#FIND count commands

In [ ]:
!find '/content/spm/spec_1024_spm/cover' -type f | wc -l


10000


**DATASET SANITY CHECK**

In [ ]:
import os
import numpy as np
from collections import defaultdict
import json
from datetime import datetime

# ============================================================================
# CONFIGURATION
# ============================================================================

# Paths to your extracted data
SPEC_1024_COVER = "/content/local_data/spec_spm_reduced/spec_1024_spm/cover"
SPEC_1024_STEGO = "/content/local_data/spec_spm_reduced/spec_1024_spm/stego"
SPEC_512_COVER = "/content/local_data/spec_spm_reduced/spec_512_spm/cover"
SPEC_512_STEGO = "/content/local_data/spec_spm_reduced/spec_512_spm/stego"

# Expected shapes (from your preprocessing)
EXPECTED_SHAPE_1024 = (513, 87)   # (freq_bins, time_frames) for n_fft=1024
EXPECTED_SHAPE_512 = (257, 173)   # (freq_bins, time_frames) for n_fft=512

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def get_base_filename(filename, spec_type="1024"):
    """Extract base name from spectrogram filename"""
    if spec_type == "1024":
        return filename.replace("_spec1024_spm.npy", "")
    else:
        return filename.replace("_spec512_spm.npy", "")


def check_npy_file(filepath):
    """
    Check if .npy file can be loaded and return stats
    Returns: (success, shape, min_val, max_val, has_nan, has_inf, error_msg)
    """
    try:
        arr = np.load(filepath)

        return (
            True,
            arr.shape,
            float(arr.min()),
            float(arr.max()),
            bool(np.isnan(arr).any()),
            bool(np.isinf(arr).any()),
            None
        )
    except Exception as e:
        return (False, None, None, None, None, None, str(e))


def analyze_directory(dir_path, spec_type="1024", expected_shape=None):
    """
    Analyze all .npy files in a directory
    Returns: dict with statistics and issues
    """
    if not os.path.exists(dir_path):
        return {"error": f"Directory not found: {dir_path}"}

    files = [f for f in os.listdir(dir_path) if f.endswith(".npy")]

    stats = {
        "total_files": len(files),
        "valid_files": 0,
        "corrupt_files": 0,
        "shape_mismatches": 0,
        "files_with_nan": 0,
        "files_with_inf": 0,
        "files_not_normalized": 0,
        "base_names": [],
        "issues": []
    }

    for fname in files:
        fpath = os.path.join(dir_path, fname)
        success, shape, min_val, max_val, has_nan, has_inf, error = check_npy_file(fpath)

        if not success:
            stats["corrupt_files"] += 1
            stats["issues"].append(f" CORRUPT: {fname} - {error}")
            continue

        stats["valid_files"] += 1
        base_name = get_base_filename(fname, spec_type)
        stats["base_names"].append(base_name)

        # Check shape
        if expected_shape and shape != expected_shape:
            stats["shape_mismatches"] += 1
            stats["issues"].append(
                f"  SHAPE MISMATCH: {fname} - Expected {expected_shape}, got {shape}"
            )

        # Check for NaN/Inf
        if has_nan:
            stats["files_with_nan"] += 1
            stats["issues"].append(f"  NaN VALUES: {fname}")

        if has_inf:
            stats["files_with_inf"] += 1
            stats["issues"].append(f"  Inf VALUES: {fname}")

        # Check normalization (should be [0, 1] after your preprocessing)
        if min_val < -0.01 or max_val > 1.01:  # Small tolerance for floating point
            stats["files_not_normalized"] += 1
            stats["issues"].append(
                f"  NOT NORMALIZED: {fname} - Range [{min_val:.4f}, {max_val:.4f}]"
            )

    return stats


# ============================================================================
# MAIN SANITY CHECK
# ============================================================================

def run_full_sanity_check():
    """
    Complete dataset validation
    """
    print("\n" + "="*80)
    print("DATASET SANITY CHECK")
    print("="*80)
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)

    # -------------------------------------------------------------------------
    # STEP 1: Check directory existence
    # -------------------------------------------------------------------------
    print("\n STEP 1: Checking directory structure...")

    dirs_to_check = {
        "1024-Cover": SPEC_1024_COVER,
        "1024-Stego": SPEC_1024_STEGO,
        "512-Cover": SPEC_512_COVER,
        "512-Stego": SPEC_512_STEGO
    }

    missing_dirs = []
    for name, path in dirs_to_check.items():
        if os.path.exists(path):
            print(f"  ✓ {name}: {path}")
        else:
            print(f"   {name}: NOT FOUND - {path}")
            missing_dirs.append(name)

    if missing_dirs:
        print(f"\n CRITICAL: Missing directories: {', '.join(missing_dirs)}")
        return

    # -------------------------------------------------------------------------
    # STEP 2: Analyze each directory
    # -------------------------------------------------------------------------
    print("\n STEP 2: Analyzing file integrity...")

    results = {}

    print("\n  Analyzing 1024-window spectrograms (Cover)...")
    results["1024_cover"] = analyze_directory(
        SPEC_1024_COVER, "1024", EXPECTED_SHAPE_1024
    )

    print("  Analyzing 1024-window spectrograms (Stego)...")
    results["1024_stego"] = analyze_directory(
        SPEC_1024_STEGO, "1024", EXPECTED_SHAPE_1024
    )

    print("  Analyzing 512-window spectrograms (Cover)...")
    results["512_cover"] = analyze_directory(
        SPEC_512_COVER, "512", EXPECTED_SHAPE_512
    )

    print("  Analyzing 512-window spectrograms (Stego)...")
    results["512_stego"] = analyze_directory(
        SPEC_512_STEGO, "512", EXPECTED_SHAPE_512
    )

    # -------------------------------------------------------------------------
    # STEP 3: Print summary statistics
    # -------------------------------------------------------------------------
    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)

    for key, stats in results.items():
        print(f"\n{key.upper().replace('_', ' - ')}:")
        print(f"  Total files: {stats['total_files']}")
        print(f"  Valid files: {stats['valid_files']}")
        print(f"  Corrupt files: {stats['corrupt_files']}")
        print(f"  Shape mismatches: {stats['shape_mismatches']}")
        print(f"  Files with NaN: {stats['files_with_nan']}")
        print(f"  Files with Inf: {stats['files_with_inf']}")
        print(f"  Files not normalized [0,1]: {stats['files_not_normalized']}")

    # -------------------------------------------------------------------------
    # STEP 4: Check for matching pairs
    # -------------------------------------------------------------------------
    print("\n" + "="*80)
    print("MATCHING PAIRS ANALYSIS")
    print("="*80)

    # Convert to sets for comparison
    cover_1024_bases = set(results["1024_cover"]["base_names"])
    stego_1024_bases = set(results["1024_stego"]["base_names"])
    cover_512_bases = set(results["512_cover"]["base_names"])
    stego_512_bases = set(results["512_stego"]["base_names"])

    print(f"\n1024-window:")
    print(f"  Cover samples: {len(cover_1024_bases)}")
    print(f"  Stego samples: {len(stego_1024_bases)}")

    missing_stego_1024 = cover_1024_bases - stego_1024_bases
    missing_cover_1024 = stego_1024_bases - cover_1024_bases

    if missing_stego_1024:
        print(f"    Cover files without stego pair: {len(missing_stego_1024)}")
        if len(missing_stego_1024) <= 10:
            print(f"      {list(missing_stego_1024)}")

    if missing_cover_1024:
        print(f"    Stego files without cover pair: {len(missing_cover_1024)}")
        if len(missing_cover_1024) <= 10:
            print(f"      {list(missing_cover_1024)}")

    if not missing_stego_1024 and not missing_cover_1024:
        print(f"  ✓ All cover/stego pairs match perfectly!")

    print(f"\n512-window:")
    print(f"  Cover samples: {len(cover_512_bases)}")
    print(f"  Stego samples: {len(stego_512_bases)}")

    missing_stego_512 = cover_512_bases - stego_512_bases
    missing_cover_512 = stego_512_bases - cover_512_bases

    if missing_stego_512:
        print(f"    Cover files without stego pair: {len(missing_stego_512)}")
        if len(missing_stego_512) <= 10:
            print(f"      {list(missing_stego_512)}")

    if missing_cover_512:
        print(f"    Stego files without cover pair: {len(missing_cover_512)}")
        if len(missing_cover_512) <= 10:
            print(f"      {list(missing_cover_512)}")

    if not missing_stego_512 and not missing_cover_512:
        print(f"  All cover/stego pairs match perfectly")

    # -------------------------------------------------------------------------
    # STEP 5: Cross-resolution matching
    # -------------------------------------------------------------------------
    print("\n" + "="*80)
    print("CROSS-RESOLUTION MATCHING")
    print("="*80)

    # For cover samples
    all_cover = cover_1024_bases & cover_512_bases
    only_1024_cover = cover_1024_bases - cover_512_bases
    only_512_cover = cover_512_bases - cover_1024_bases

    print(f"\nCover samples:")
    print(f"  Matching in both resolutions: {len(all_cover)}")
    if only_1024_cover:
        print(f"    Only in 1024: {len(only_1024_cover)}")
    if only_512_cover:
        print(f"    Only in 512: {len(only_512_cover)}")

    # For stego samples
    all_stego = stego_1024_bases & stego_512_bases
    only_1024_stego = stego_1024_bases - stego_512_bases
    only_512_stego = stego_512_bases - stego_1024_bases

    print(f"\nStego samples:")
    print(f"  Matching in both resolutions: {len(all_stego)}")
    if only_1024_stego:
        print(f"    Only in 1024: {len(only_1024_stego)}")
    if only_512_stego:
        print(f"    Only in 512: {len(only_512_stego)}")

    # -------------------------------------------------------------------------
    # STEP 6: Final usable dataset size
    # -------------------------------------------------------------------------
    print("\n" + "="*80)
    print("FINAL USABLE DATASET")
    print("="*80)

    # Samples that exist in ALL four directories
    usable_samples = all_cover & all_stego

    print(f"\nSamples with complete data (both resolutions + both classes):")
    print(f"  Total usable pairs: {len(usable_samples)}")
    print(f"  Total usable samples: {len(usable_samples) * 2} (cover + stego)")

    if len(usable_samples) == 0:
        print("\n CRITICAL: No usable samples found!")
    elif len(usable_samples) < 100:
        print("\n  WARNING: Very small dataset! Consider preprocessing more samples.")
    else:
        print("\n✓ Dataset looks good!")

    # -------------------------------------------------------------------------
    # STEP 7: Print detailed issues
    # -------------------------------------------------------------------------
    total_issues = sum(len(r["issues"]) for r in results.values())

    if total_issues > 0:
        print("\n" + "="*80)
        print(f"DETAILED ISSUES ({total_issues} total)")
        print("="*80)

        for key, stats in results.items():
            if stats["issues"]:
                print(f"\n{key.upper().replace('_', ' - ')}:")
                for issue in stats["issues"][:20]:  # Show max 20 issues per category
                    print(f"  {issue}")

                if len(stats["issues"]) > 20:
                    print(f"  ... and {len(stats['issues']) - 20} more issues")

    # -------------------------------------------------------------------------
    # STEP 8: Sample data check
    # -------------------------------------------------------------------------
    print("\n" + "="*80)
    print("SAMPLE DATA VERIFICATION")
    print("="*80)

    if len(usable_samples) > 0:
        sample_base = list(usable_samples)[0]
        print(f"\nLoading sample: {sample_base}")

        try:
            cover_1024 = np.load(os.path.join(SPEC_1024_COVER, f"{sample_base}_spec1024_spm.npy"))
            stego_1024 = np.load(os.path.join(SPEC_1024_STEGO, f"{sample_base}_spec1024_spm.npy"))
            cover_512 = np.load(os.path.join(SPEC_512_COVER, f"{sample_base}_spec512_spm.npy"))
            stego_512 = np.load(os.path.join(SPEC_512_STEGO, f"{sample_base}_spec512_spm.npy"))

            print(f"\n  ✓ All files loaded successfully")
            print(f"\n  Shapes:")
            print(f"    Cover 1024: {cover_1024.shape}")
            print(f"    Stego 1024: {stego_1024.shape}")
            print(f"    Cover 512:  {cover_512.shape}")
            print(f"    Stego 512:  {stego_512.shape}")

            print(f"\n  Value ranges:")
            print(f"    Cover 1024: [{cover_1024.min():.4f}, {cover_1024.max():.4f}]")
            print(f"    Stego 1024: [{stego_1024.min():.4f}, {stego_1024.max():.4f}]")
            print(f"    Cover 512:  [{cover_512.min():.4f}, {cover_512.max():.4f}]")
            print(f"    Stego 512:  [{stego_512.min():.4f}, {stego_512.max():.4f}]")

            # Check if cover and stego are different (should be!)
            diff_1024 = np.abs(cover_1024 - stego_1024).mean()
            diff_512 = np.abs(cover_512 - stego_512).mean()

            print(f"\n  Mean absolute difference (Cover vs Stego):")
            print(f"    1024-window: {diff_1024:.6f}")
            print(f"    512-window:  {diff_512:.6f}")

            if diff_1024 < 1e-6 or diff_512 < 1e-6:
                print(f"\n    WARNING: Cover and stego are nearly identical!")
                print(f"      This suggests preprocessing error or wrong file pairing.")
            else:
                print(f"\n  ✓ Cover and stego show expected differences")

        except Exception as e:
            print(f"\n   Error loading sample: {e}")

    # -------------------------------------------------------------------------
    # FINAL VERDICT
    # -------------------------------------------------------------------------
    print("\n" + "="*80)
    print("FINAL VERDICT")
    print("="*80)

    critical_issues = []
    warnings = []

    # Check for critical issues
    if len(usable_samples) == 0:
        critical_issues.append("No usable samples found")

    for key, stats in results.items():
        if stats["corrupt_files"] > 0:
            critical_issues.append(f"{key}: {stats['corrupt_files']} corrupt files")
        if stats["files_with_nan"] > 0:
            warnings.append(f"{key}: {stats['files_with_nan']} files with NaN values")
        if stats["files_with_inf"] > 0:
            warnings.append(f"{key}: {stats['files_with_inf']} files with Inf values")

    if critical_issues:
        print("\n CRITICAL ISSUES FOUND:")
        for issue in critical_issues:
            print(f"  • {issue}")
        print("\n  → You MUST fix these before training!")

    if warnings:
        print("\n  WARNINGS:")
        for warning in warnings:
            print(f"  • {warning}")
        print("\n  → These should be investigated")

    if not critical_issues and not warnings:
        print("\n ALL CHECKS PASSED!")
        print(f"\n  Your dataset is ready for training with {len(usable_samples)} samples")
        print(f"  Total: {len(usable_samples) * 2} samples (cover + stego)")

    print("\n" + "="*80)

    return results, usable_samples


# ============================================================================
# RUN THE CHECK
# ============================================================================

if __name__ == "__main__":
    results, usable_samples = run_full_sanity_check()

#Copy files from drive to colab



In [ ]:
!mkdir /content/spm/

In [ ]:
!cp -r "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/spec_512_spm" \
      /content/spm/spec_512_spm


#Dataset splitting

In [ ]:
from torch.utils.data import Dataset, ConcatDataset, Subset
from sklearn.model_selection import train_test_split
import os
import numpy as np
import torch
import random
import json
from datetime import datetime
from torch.utils.data import DataLoader


# Create a link to data directory
source_dir = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/indices_v2"

# ============================================================================
# DATASET CLASS
# ============================================================================

class SpecDataset(Dataset):
    def __init__(self, spec1024_folder, spec512_folder, label):
        self.spec1024_folder = spec1024_folder
        self.spec512_folder = spec512_folder
        self.label = label

        # Verify folders exist
        if not os.path.exists(spec1024_folder):
            raise FileNotFoundError(f"1024 folder not found: {spec1024_folder}")
        if not os.path.exists(spec512_folder):
            raise FileNotFoundError(f"512 folder not found: {spec512_folder}")

        files_1024 = sorted(os.listdir(spec1024_folder))
        files_512  = sorted(os.listdir(spec512_folder))

        # Extract base names
        base_1024 = [f.replace("_spec1024_spm.npy", "") for f in files_1024 if f.endswith("_spec1024_spm.npy")]
        base_512  = [f.replace("_spec512_spm.npy", "")  for f in files_512 if f.endswith("_spec512_spm.npy")]

        # Find matching base names
        matching_bases = list(set(base_1024) & set(base_512))
        matching_bases.sort()

        if len(matching_bases) == 0:
            raise ValueError("No matching files found in both folders!")

        # Store full paths
        self.files_1024 = [os.path.join(spec1024_folder, f"{b}_spec1024_spm.npy") for b in matching_bases]
        self.files_512  = [os.path.join(spec512_folder,  f"{b}_spec512_spm.npy") for b in matching_bases]

    def __len__(self):
        return len(self.files_1024)

    def __getitem__(self, idx):
    # Load spectrograms safely (allow_pickle only if necessary)
      try:
        spec1024 = np.load(self.files_1024[idx], allow_pickle=False)
      except ValueError:
        print(f" Pickled data detected in {self.files_1024[idx]}, loading with allow_pickle=True")
        spec1024 = np.load(self.files_1024[idx], allow_pickle=True)

      try:
        spec512 = np.load(self.files_512[idx], allow_pickle=False)
      except ValueError:
        print(f" Pickled data detected in {self.files_512[idx]}, loading with allow_pickle=True")
        spec512 = np.load(self.files_512[idx], allow_pickle=True)

      # Convert to torch tensors and add channel dim (1, F, T)
      spec1024 = torch.tensor(spec1024, dtype=torch.float32).unsqueeze(0)
      spec512  = torch.tensor(spec512, dtype=torch.float32).unsqueeze(0)

      # Use long (integer) type for classification labels
      label = torch.tensor(self.label, dtype=torch.long)

      return spec1024, spec512, label



def set_all_seeds(seed=42):
    """
    Fix all random seeds for reproducibility across ablation studies.
    Critical for comparing model variants fairly.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"✓ All seeds set to {seed} for reproducibility")


def create_stratified_splits(cover_dataset, stego_dataset,
                             train_ratio=0.7, val_ratio=0.15, test_ratio=0.15,
                             random_seed=42, save_indices=True,
                             experiment_name="baseline"):

    # Verify ratios sum to 1
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, \
        "Split ratios must sum to 1.0"

    # Check dataset balance (critical for steganalysis)
    cover_size = len(cover_dataset)
    stego_size = len(stego_dataset)

    print("\n" + "="*60)
    print("DATASET BALANCE CHECK")
    print("="*60)
    print(f"Cover samples: {cover_size}")
    print(f"Stego samples: {stego_size}")

    if cover_size != stego_size:
        imbalance_ratio = max(cover_size, stego_size) / min(cover_size, stego_size)
        print(f"  WARNING: Class imbalance detected (ratio: {imbalance_ratio:.2f}:1)")
        print("   Consider balancing or using weighted loss functions")
    else:
        print("✓ Datasets are perfectly balanced")

    # Combine datasets
    full_dataset = ConcatDataset([cover_dataset, stego_dataset])
    total_size = len(full_dataset)

    # Get indices for each class
    cover_indices = list(range(cover_size))
    stego_indices = list(range(cover_size, total_size))

    # Set seed for reproducibility
    set_all_seeds(random_seed)

    # Stratified split for cover samples
    cover_train, cover_temp = train_test_split(
        cover_indices,
        test_size=(val_ratio + test_ratio),
        random_state=random_seed
    )
    cover_val, cover_test = train_test_split(
        cover_temp,
        test_size=(test_ratio / (val_ratio + test_ratio)),
        random_state=random_seed
    )

    # Stratified split for stego samples
    stego_train, stego_temp = train_test_split(
        stego_indices,
        test_size=(val_ratio + test_ratio),
        random_state=random_seed
    )
    stego_val, stego_test = train_test_split(
        stego_temp,
        test_size=(test_ratio / (val_ratio + test_ratio)),
        random_state=random_seed
    )

    # Combine and shuffle indices
    train_indices = cover_train + stego_train
    val_indices = cover_val + stego_val
    test_indices = cover_test + stego_test

    random.shuffle(train_indices)
    random.shuffle(val_indices)
    random.shuffle(test_indices)

    # Create subsets
    train_dataset = Subset(full_dataset, train_indices)
    val_dataset = Subset(full_dataset, val_indices)
    test_dataset = Subset(full_dataset, test_indices)

    # Calculate class distribution in each split
    def count_classes(indices, cover_size):
        cover_count = sum(1 for i in indices if i < cover_size)
        stego_count = len(indices) - cover_count
        return cover_count, stego_count

    train_cover, train_stego = count_classes(train_indices, cover_size)
    val_cover, val_stego = count_classes(val_indices, cover_size)
    test_cover, test_stego = count_classes(test_indices, cover_size)

    # Print split statistics
    print("\n" + "="*60)
    print("SPLIT STATISTICS")
    print("="*60)
    print(f"Total samples: {total_size}")
    print(f"\nTrain set: {len(train_dataset)} samples ({train_ratio*100:.1f}%)")
    print(f"  ├─ Cover: {train_cover}")
    print(f"  └─ Stego: {train_stego}")
    print(f"\nValidation set: {len(val_dataset)} samples ({val_ratio*100:.1f}%)")
    print(f"  ├─ Cover: {val_cover}")
    print(f"  └─ Stego: {val_stego}")
    print(f"\nTest set: {len(test_dataset)} samples ({test_ratio*100:.1f}%)")
    print(f"  ├─ Cover: {test_cover}")
    print(f"  └─ Stego: {test_stego}")
    print("="*60)

    # Prepare split info for saving
    split_info = {
        "experiment_name": experiment_name,
        "timestamp": datetime.now().isoformat(),
        "random_seed": random_seed,
        "total_samples": total_size,
        "cover_samples": cover_size,
        "stego_samples": stego_size,
        "split_ratios": {
            "train": train_ratio,
            "val": val_ratio,
            "test": test_ratio
        },
        "split_sizes": {
            "train": len(train_dataset),
            "val": len(val_dataset),
            "test": len(test_dataset)
        },
        "class_distribution": {
            "train": {"cover": train_cover, "stego": train_stego},
            "val": {"cover": val_cover, "stego": val_stego},
            "test": {"cover": test_cover, "stego": test_stego}
        },
        "indices": {
            "train": train_indices,
            "val": val_indices,
            "test": test_indices
        }
    }

    # Save indices to disk for reproducibility
    if save_indices:
        # Save to both Drive and local for safety
        save_path_drive = f"{source_dir}/split_indices_{experiment_name}_seed{random_seed}.json"
        save_path_local = f"/content/split_indices_{experiment_name}_seed{random_seed}.json"

        # Convert numpy types to native Python types for JSON serialization
        split_info_serializable = {
            k: (v.tolist() if isinstance(v, np.ndarray) else
                {k2: (v2.tolist() if isinstance(v2, np.ndarray) else v2)
                 for k2, v2 in v.items()} if isinstance(v, dict) else v)
            for k, v in split_info.items()
        }

        # Save to local first (fast)
        with open(save_path_local, 'w') as f:
            json.dump(split_info_serializable, f, indent=2)
        print(f"\n Split indices saved locally: {save_path_local}")

        # Try to save to Drive (may timeout, but we have local copy)
        try:
            with open(save_path_drive, 'w') as f:
                json.dump(split_info_serializable, f, indent=2)
            print(f" Split indices backed up to Drive: {save_path_drive}")
        except Exception as e:
            print(f"  Could not save to Drive (using local copy): {e}")

        print("  Use these SAME indices for all ablation experiments!")

    return train_dataset, val_dataset, test_dataset, split_info


def load_split_indices(filepath):
    """
    Load previously saved split indices to ensure consistency across experiments.
    Critical for fair ablation study comparisons.
    """
    with open(filepath, 'r') as f:
        split_info = json.load(f)
    print(f"✓ Loaded split indices from: {filepath}")
    print(f"  Experiment: {split_info['experiment_name']}")
    print(f"  Seed: {split_info['random_seed']}")
    return split_info


# ============================================================================
# MAIN EXECUTION
# ============================================================================

# Set global seed (do only once at the start)
set_all_seeds(seed=42)

print("\n" + "="*60)
print("LOADING DATASETS")
print("="*60)

cover_dataset = SpecDataset(
    spec1024_folder="/content/spm/spec_1024_spm/cover",
    spec512_folder="/content/spm/spec_512_spm/cover",
    label=0
)

stego_dataset = SpecDataset(
    spec1024_folder="/content/spm/spec_1024_spm/stego",
    spec512_folder="/content/spm/spec_512_spm/stego",
    label=1
)


print(" Datasets loaded successfully")

SPLIT_PATH = f"{source_dir}/split_indices_baseline_full_model_seed42.json"

full_dataset = ConcatDataset([cover_dataset, stego_dataset])

if os.path.exists(SPLIT_PATH):
    print("\n Using existing fixed split indices")

    split_info = load_split_indices(SPLIT_PATH)

    train_dataset = Subset(full_dataset, split_info["indices"]["train"])
    val_dataset   = Subset(full_dataset, split_info["indices"]["val"])
    test_dataset  = Subset(full_dataset, split_info["indices"]["test"])

else:
    print("\n  No existing split found. Creating new one (ONLY DO THIS ONCE).")

    train_dataset, val_dataset, test_dataset, split_info = create_stratified_splits(
        cover_dataset=cover_dataset,
        stego_dataset=stego_dataset,
        train_ratio=0.7,
        val_ratio=0.15,
        test_ratio=0.15,
        random_seed=42,
        save_indices=True,
        experiment_name="baseline_full_model"
    )


# Verify a sample can be loaded
print("\n" + "="*60)
print("SAMPLE VERIFICATION")
print("="*60)
spec1024, spec512, label = train_dataset[0]
print(f"Sample shapes:")
print(f"  ├─ Spec1024: {spec1024.shape}")
print(f"  ├─ Spec512: {spec512.shape}")
print(f"  └─ Label: {label.item()}")
print("="*60)


print("\n" + "="*60)
print("CREATING DATALOADERS")
print("="*60)

BATCH_SIZE = 64
NUM_WORKERS = 0
PIN_MEMORY = True

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print(f"✓ DataLoaders created:")
print(f"  ├─ Train batches: {len(train_loader)}")
print(f"  ├─ Val batches: {len(val_loader)}")
print(f"  └─ Test batches: {len(test_loader)}")
print("="*60)

print("\n✓ Dataset splitting complete and ready for ablation studies!")
print("  Remember: Use the SAME split indices for ALL experiments!")

✓ All seeds set to 42 for reproducibility

LOADING DATASETS
✓ Datasets loaded successfully

✓ Using existing fixed split indices
✓ Loaded split indices from: /content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/indices_v2/split_indices_baseline_full_model_seed42.json
  Experiment: baseline_full_model
  Seed: 42

SAMPLE VERIFICATION
Sample shapes:
  ├─ Spec1024: torch.Size([1, 513, 87])
  ├─ Spec512: torch.Size([1, 257, 173])
  └─ Label: 1

CREATING DATALOADERS
✓ DataLoaders created:
  ├─ Train batches: 219
  ├─ Val batches: 47
  └─ Test batches: 47

✓ Dataset splitting complete and ready for ablation studies!
  Remember: Use the SAME split indices for ALL experiments!


**FULL MODEL**


**Spectogram feature extraction**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =====================================================
# RESIDUAL UNIT (SPM-SAFE)
# =====================================================

class ImprovedResidualUnit(nn.Module):
    """Better residual unit with batch norm"""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return F.relu(out)


class ImprovedSResNetStream(nn.Module):
    """
    Deeper feature extractor for steganalysis
    """
    def __init__(self, input_channels=1):
        super().__init__()

        # Stage 1: Initial feature extraction
        self.conv1 = nn.Conv2d(input_channels, 16, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.res1 = nn.Sequential(
            ImprovedResidualUnit(16),
            ImprovedResidualUnit(16)
        )

        # Stage 2: Downsample + more features
        self.conv2 = nn.Conv2d(16, 32, 3, stride=2, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(32)
        self.res2 = nn.Sequential(
            ImprovedResidualUnit(32),
            ImprovedResidualUnit(32)
        )

        # Stage 3: Further downsampling
        self.conv3 = nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(64)
        self.res3 = nn.Sequential(
            ImprovedResidualUnit(64),
            ImprovedResidualUnit(64)
        )

        # Stage 4: Final feature compression
        self.conv4 = nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False)
        self.bn4 = nn.BatchNorm2d(128)
        self.res4 = ImprovedResidualUnit(128)

    def forward(self, x):
        # Stage 1
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.res1(x)

        # Stage 2
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.res2(x)

        # Stage 3
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.res3(x)

        # Stage 4
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.res4(x)

        # Global pooling
        x = F.adaptive_avg_pool2d(x, (1, 1))
        return x.view(x.size(0), -1)  # (batch, 128)


class ImprovedDualStreamExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.shared_stream = ImprovedSResNetStream()

    def forward(self, spec1024, spec512):
        f1024 = self.shared_stream(spec1024)
        f512  = self.shared_stream(spec512)
        return f1024, f512



**Attention feature fusion**

In [ ]:
# =====================================================
# EXPLAINABLE SCALAR ATTENTION
# =====================================================

class ImprovedScalarAttentionFusion(nn.Module):
    """
    Fixed attention with proper gradient flow
    """
    def __init__(self):
        super().__init__()
        # Learnable parameters
        self.alpha = nn.Parameter(torch.ones(2))

    def forward(self, f1024, f512):
        # Apply softmax to get normalized weights
        weights = F.softmax(self.alpha, dim=0)

        # Element-wise weighted sum (better than concatenation)
        fused = weights[0] * f1024 + weights[1] * f512

        attention_info = {
            'alpha_1024': weights[0].detach(),
            'alpha_512':  weights[1].detach()
        }

        return fused, attention_info


**Lightweight MLP classifier**

In [ ]:
# =====================================================
# LIGHTWEIGHT CLASSIFIER
# =====================================================

class ImprovedClassifier(nn.Module):
    def __init__(self, input_dim=128):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.fc(x)


**Full pipeline model**

In [ ]:
# =====================================================
# FULL UNIVERSAL EXPLAINABLE MODEL
# =====================================================

class ImprovedExplainableSteganalysisModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.extractor = ImprovedDualStreamExtractor()
        self.fusion = ImprovedScalarAttentionFusion()
        self.classifier = ImprovedClassifier(input_dim=128)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_normal_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward(self, spec1024, spec512, return_explanations=False):
        f1024, f512 = self.extractor(spec1024, spec512)
        fused, attn = self.fusion(f1024, f512)
        logits = self.classifier(fused)

        if return_explanations:
            return logits, {
                'stream_attention': attn,
                'features': {
                    'f1024': f1024.detach().cpu(),
                    'f512': f512.detach().cpu()
                },
                'probabilities': F.softmax(logits, dim=1).detach().cpu()
            }

        return logits, attn



In [ ]:

# =====================================================
# HELPER: Visualize Explanations
# =====================================================

def visualize_explanation(model, spec1024, spec512, label, idx=0):
    """
    Call after training to inspect model explanations
    """
    model.eval()
    with torch.no_grad():
        logits, explanations = model(
            spec1024,
            spec512,
            return_explanations=True
        )

    print(f"\n=== Explanation for Sample {idx} ===")
    print(f"True Label: {label[idx].item()}")

    probs = explanations["decision"]["probabilities"][idx]
    pred = probs.argmax().item()

    print(
        f"Predicted: {pred} "
        f"(Cover: {probs[0]:.3f}, Stego: {probs[1]:.3f})"
    )

    alpha_1024 = explanations["stream_attention"]["alpha_1024"][idx].item()
    alpha_512  = explanations["stream_attention"]["alpha_512"][idx].item()

    print("\nStream Importance:")
    print(f"  1024-FFT: {alpha_1024:.3f}")
    print(f"  512-FFT:  {alpha_512:.3f}")

    if alpha_1024 > alpha_512:
        print("  → Model relied more on HIGH-frequency information")
    else:
        print("  → Model relied more on LOW-frequency information")

    return explanations

#Training per epoch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
import os
import glob
import pandas as pd
import math
from collections import defaultdict
import time
import platform
from tqdm import tqdm

# ---- Device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---- Config ----
num_epochs = 50
patience = 10
lr = 1e-4
batch_size = 64
save_every = 500
optimizer_name = "AdamW"
gradient_clip_max_norm = 1.0  # Added gradient clipping

checkpoint_dir = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Models/backbone_model_v6"
os.makedirs(checkpoint_dir, exist_ok=True)

results_file = os.path.join(checkpoint_dir, "experiment_results.csv")
per_seed_history_dir = os.path.join(checkpoint_dir, "per_seed_histories")
os.makedirs(per_seed_history_dir, exist_ok=True)

# =====================================================
# DEBUGGING FUNCTIONS
# =====================================================

def verify_data_loader(loader, name="DataLoader"):
    """Verify that data loader is properly configured"""
    print(f"\n=== Verifying {name} ===")

    try:
        spec1024, spec512, labels = next(iter(loader))
        print(f"✓ Batch loading successful")
        print(f"  Batch size: {spec1024.size(0)}")
        print(f"  spec1024 shape: {spec1024.shape}")
        print(f"  spec512 shape: {spec512.shape}")
        print(f"  Labels shape: {labels.shape}")

        # Check label distribution
        unique, counts = np.unique(labels.numpy(), return_counts=True)
        print(f"  Label distribution:")
        for u, c in zip(unique, counts):
            print(f"    Class {u}: {c} ({c/len(labels)*100:.1f}%)")

        # Check for class imbalance
        if len(unique) < 2:
            print(f"    WARNING: Only one class in batch! This will cause training failure.")
            return False

        # Check data range
        print(f"  spec1024 range: [{spec1024.min():.3f}, {spec1024.max():.3f}]")
        print(f"  spec512 range: [{spec512.min():.3f}, {spec512.max():.3f}]")

        # Check for anomalies
        if torch.isnan(spec1024).any() or torch.isinf(spec1024).any():
            print(f"    WARNING: spec1024 contains NaN or Inf!")
            return False
        if torch.isnan(spec512).any() or torch.isinf(spec512).any():
            print(f"    WARNING: spec512 contains NaN or Inf!")
            return False

        return True

    except Exception as e:
        print(f"  ✗ Error loading batch: {e}")
        return False


def debug_first_batch(batch_idx, spec1024, spec512, label, model):
    """Debug function to check if data is valid"""
    print("\n=== DEBUGGING FIRST BATCH ===")
    print(f"  spec1024 shape: {spec1024.shape}, range: [{spec1024.min():.3f}, {spec1024.max():.3f}]")
    print(f"  spec512 shape: {spec512.shape}, range: [{spec512.min():.3f}, {spec512.max():.3f}]")
    print(f"  labels: {label.cpu().numpy()[:10]}...")
    print(f"  label distribution: {np.bincount(label.cpu().numpy().astype(int))}")

    # Check model output
    model.eval()
    with torch.no_grad():
        s1024 = spec1024[:2].to(device)
        s512 = spec512[:2].to(device)
        logits, alpha = model(s1024, s512)
        print(f"  Model output logits: {logits}")
        print(f"  Model output probs: {torch.softmax(logits, dim=1)}")
        if isinstance(alpha, dict):
            for k, v in alpha.items():
                print(f"  {k}: {v.item():.4f}")
        else:
            print(f"  Attention alpha: {alpha}")
    model.train()
    print("======================\n")

# ---- Helper: check if seed completed ----
def is_seed_completed(seed, variant_name, results_file):
    """Check if this seed already completed training"""
    if not os.path.exists(results_file):
        return False

    try:
        df = pd.read_csv(results_file)
        completed = df[(df["variant"] == variant_name) & (df["seed"] == seed)]

        if len(completed) > 0:
            row = completed.iloc[-1]
            if row["early_stopped"] or row["total_epochs_run"] >= num_epochs:
                return True
    except Exception as e:
        print(f"Warning: Could not check completion status: {e}")

    return False

# ---- Helper: find latest checkpoint ----
def get_latest_checkpoint(seed, variant_name):
    """Find the latest checkpoint by parsing epoch number"""
    pattern = os.path.join(checkpoint_dir, f"{variant_name}_seed{seed}_epoch*.pt")
    files = glob.glob(pattern)

    if not files:
        return None

    # Parse epoch from filenames to find the truly latest one
    latest_file = None
    max_epoch = -1

    for f in files:
        try:
            basename = os.path.basename(f)
            # Extract epoch number
            parts = basename.replace(".pt", "").split("_")
            epoch_str = [p for p in parts if p.startswith("epoch")][0]
            epoch_num = int(epoch_str.replace("epoch", ""))

            if epoch_num > max_epoch:
                max_epoch = epoch_num
                latest_file = f
        except:
            continue

    return latest_file

# ---- Reproducibility ----
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ---- Class balance check ----
def compute_class_distribution(loader, name="Loader"):
    all_labels = []
    for _, _, labels in loader:
        all_labels.extend(labels.numpy())
    unique, counts = np.unique(all_labels, return_counts=True)
    print(f"{name} class distribution:")
    for u, c in zip(unique, counts):
        print(f"  Class {u}: {c} samples ({c/len(all_labels):.2%})")

# ---- Utility: best threshold by maximizing F1 on validation ----
def find_best_threshold(labels, probs, step=0.01):
    best_thr = 0.5
    best_f1 = -1
    labels = np.array(labels)
    probs = np.array(probs)
    if len(np.unique(labels)) == 1:
        return best_thr, best_f1
    for thr in np.arange(0.0, 1.0001, step):
        preds = (probs >= thr).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr
    return best_thr, best_f1

# ---- Utility: 95% CI for mean ----
def mean_std_ci(arr, conf=0.95):
    arr = np.array(arr)
    if len(arr) == 0:
        return (np.nan, np.nan, np.nan)
    mean = np.nanmean(arr)
    std = np.nanstd(arr, ddof=1) if len(arr) > 1 else 0.0
    se = std / math.sqrt(len(arr)) if len(arr) > 1 else 0.0
    z = 1.96
    ci = z * se
    return mean, std, ci

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# =====================================================
# TRAINING FUNCTION WITH EPOCH-LEVEL CHECKPOINTS
# =====================================================

def train_and_validate_variant(seed, variant_name, variant_constructor, train_loader, val_loader,
                                test_loader=None, er_bin="NA", recomp_test_loader=None):
    """
    Enhanced training function with epoch-level checkpoint saving and resuming
    """
    start_time = time.time()

    set_seed(seed)
    model = variant_constructor().to(device)

    n_params = count_parameters(model)
    print(f"\n Model: {variant_name}")
    print(f"   Parameters: {n_params:,}")
    print(f"   Size: {sum(p.numel() * p.element_size() for p in model.parameters()) / (1024**2):.2f} MB")

    # Initialize optimizer and scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4,
        betas=(0.9, 0.999)
    )

    criterion = nn.CrossEntropyLoss()

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=5,
    )

    # Training state
    start_epoch = 0
    best_val_f1 = -1
    best_val_auc = -1
    best_epoch = 0
    early_stopped = False
    patience_counter = 0
    best_ckpt_path = None

    # History tracking
    history = {
        "train_loss": [], "train_acc": [],
        "val_loss": [], "val_acc": [], "val_prec": [],
        "val_rec": [], "val_f1": [], "val_auc": [],
        "lr": []
    }

    # ===== LOAD CHECKPOINT IF EXISTS =====
    latest_ckpt = get_latest_checkpoint(seed, variant_name)
    if latest_ckpt:
        print(f"\n Found checkpoint: {latest_ckpt}")
        try:
            ckpt = torch.load(latest_ckpt, map_location=device, weights_only=False)

            model.load_state_dict(ckpt['model_state_dict'])
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])

            if 'scheduler_state_dict' in ckpt:
                scheduler.load_state_dict(ckpt['scheduler_state_dict'])

            start_epoch = ckpt.get('epoch', 0)
            best_val_f1 = ckpt.get('best_val_f1', -1)
            best_val_auc = ckpt.get('best_val_auc', -1)
            patience_counter = ckpt.get('patience_counter', 0)

            # Load history if available
            if 'history' in ckpt:
                history = ckpt['history']

            print(f" Resumed from epoch {start_epoch}")
            print(f"   Best Val F1: {best_val_f1:.4f}, Patience: {patience_counter}/{patience}")

        except Exception as e:
            print(f"  Could not load checkpoint: {e}")
            print("   Starting from scratch...")
            start_epoch = 0

    # ===== TRAINING LOOP =====
    for epoch in range(start_epoch, num_epochs):
        epoch_start = time.time()

        # ===== TRAINING PHASE =====
        model.train()
        total_loss = 0
        train_preds_all = []
        train_labels_all = []

        # Create progress bar
        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                    desc=f"Epoch {epoch+1}/{num_epochs}")

        for batch_idx, (spec1024, spec512, label) in pbar:

            spec1024, spec512, label = spec1024.to(device), spec512.to(device), label.to(device)

            optimizer.zero_grad()

            # Forward pass
            logits, alpha = model(spec1024, spec512)
            loss = criterion(logits, label.long())

            # Check for NaN
            if torch.isnan(loss):
                print(f"  NaN loss detected at epoch {epoch+1}, batch {batch_idx}")
                print(f"   Skipping batch...")
                continue

            # Backward pass with gradient clipping
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip_max_norm)
            optimizer.step()

            total_loss += loss.item()

            # Track training accuracy
            preds = logits.argmax(dim=1).cpu().numpy()
            train_preds_all.extend(preds)
            train_labels_all.extend(label.cpu().numpy())

            # Update progress bar
            if len(train_preds_all) > 0:
                current_acc = accuracy_score(train_labels_all, train_preds_all)
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{current_acc:.4f}'
                })

        train_loss = total_loss / len(train_loader)
        train_acc = accuracy_score(train_labels_all, train_preds_all)

        # ===== VALIDATION PHASE =====
        model.eval()
        val_labels, val_preds, val_probs = [], [], []
        val_loss = 0.0

        with torch.no_grad():
            for spec1024, spec512, label in tqdm(val_loader, desc="Validating"):
                spec1024, spec512, label = spec1024.to(device), spec512.to(device), label.to(device)

                logits, alpha = model(spec1024, spec512)
                loss = criterion(logits, label.long())
                val_loss += loss.item()

                probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                preds = logits.argmax(dim=1).cpu().numpy()

                val_labels.extend(label.cpu().numpy())
                val_preds.extend(preds)
                val_probs.extend(probs)

        # Compute metrics
        acc = accuracy_score(val_labels, val_preds)
        precision = precision_score(val_labels, val_preds, zero_division=0)
        recall = recall_score(val_labels, val_preds, zero_division=0)
        f1 = f1_score(val_labels, val_preds, zero_division=0)

        try:
            auc = roc_auc_score(val_labels, val_probs)
        except:
            auc = float("nan")

        # Update history
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss / len(val_loader))
        history["val_acc"].append(acc)
        history["val_prec"].append(precision)
        history["val_rec"].append(recall)
        history["val_f1"].append(f1)
        history["val_auc"].append(auc)
        history["lr"].append(optimizer.param_groups[0]['lr'])

        epoch_time = time.time() - epoch_start

        # Print epoch summary
        print(f"\n{'='*80}")
        print(f"[{variant_name}][Seed {seed}] Epoch {epoch+1}/{num_epochs} ({epoch_time:.1f}s)")
        print(f"  Train: Loss={train_loss:.4f}, Acc={train_acc:.4f}")
        print(f"  Val:   Loss={val_loss/len(val_loader):.4f}, Acc={acc:.4f}, Prec={precision:.4f}, Rec={recall:.4f}, F1={f1:.4f}, AUC={auc:.4f}")
        print(f"  LR: {optimizer.param_groups[0]['lr']:.2e}")

        # Update learning rate
        scheduler.step(f1)

        # Save best model
        if f1 > best_val_f1:
            best_val_f1 = f1
            best_val_auc = auc
            best_epoch = epoch + 1
            patience_counter = 0

            best_ckpt_path = os.path.join(
                checkpoint_dir,
                f"{variant_name}_seed{seed}_best_F1{f1:.4f}.pt"
            )
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'patience_counter': patience_counter,
                'val_f1': f1,
                'val_auc': auc,
                'best_val_f1': f1,
                'best_val_auc': auc,
                'history': history,
            }, best_ckpt_path)
            print(f" New best model saved: {best_ckpt_path} (Val F1: {f1:.4f}, AUC: {auc:.4f})")
        else:
            patience_counter += 1
            print(f"  Patience: {patience_counter}/{patience}")

        # Save checkpoint at end of each epoch
        epoch_ckpt_path = os.path.join(
            checkpoint_dir,
            f"{variant_name}_seed{seed}_epoch{epoch+1}.pt"
        )

        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'patience_counter': patience_counter,
            'val_f1': f1,
            'val_auc': auc,
            'best_val_f1': best_val_f1,
            'best_val_auc': best_val_auc,
            'history': history,
        }, epoch_ckpt_path)

        if patience_counter >= patience:
                print(f" Early stopping at epoch {epoch+1}")
                early_stopped = True
                break

        print(f"{'='*80}\n")

    # ===== SAVE HISTORY =====
    hist_df = pd.DataFrame(history)
    hist_csv_path = os.path.join(per_seed_history_dir, f"{variant_name}_seed{seed}_history.csv")
    hist_df.to_csv(hist_csv_path, index=False)

    # ===== FIND BEST THRESHOLD ON VALIDATION =====
    if best_ckpt_path and os.path.exists(best_ckpt_path):
        ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])

    val_labels_all, val_probs_all = [], []
    model.eval()
    with torch.no_grad():
        for spec1024, spec512, label in val_loader:
            spec1024, spec512, label = spec1024.to(device), spec512.to(device), label.to(device)
            logits, _ = model(spec1024, spec512)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            val_probs_all.extend(probs)
            val_labels_all.extend(label.cpu().numpy())

    val_selected_threshold, val_thr_f1 = find_best_threshold(val_labels_all, val_probs_all, step=0.01)

    # ===== TEST SET EVALUATION =====
    test_metrics = {"test_f1": np.nan, "test_acc": np.nan, "test_prec": np.nan, "test_rec": np.nan}
    if test_loader is not None:
        test_labels, test_probs, test_preds = [], [], []
        with torch.no_grad():
            for spec1024, spec512, label in tqdm(test_loader, desc="Testing"):
                spec1024, spec512, label = spec1024.to(device), spec512.to(device), label.to(device)
                logits, _ = model(spec1024, spec512)
                probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                preds = (probs >= val_selected_threshold).astype(int)
                test_labels.extend(label.cpu().numpy())
                test_probs.extend(probs)
                test_preds.extend(preds)

        test_f1 = f1_score(test_labels, test_preds, zero_division=0)
        test_acc = accuracy_score(test_labels, test_preds)
        test_prec = precision_score(test_labels, test_preds, zero_division=0)
        test_rec = recall_score(test_labels, test_preds, zero_division=0)
        test_metrics = {"test_f1": test_f1, "test_acc": test_acc, "test_prec": test_prec, "test_rec": test_rec}

        print(f"\n Test Set Results:")
        print(f"  Acc: {test_acc:.4f}, F1: {test_f1:.4f}, Prec: {test_prec:.4f}, Rec: {test_rec:.4f}")

    # ===== ROBUSTNESS EVALUATION =====
    robustness = {"recomp_test_f1": np.nan, "delta_f1": np.nan}
    if recomp_test_loader is not None:
        recomp_labels, recomp_preds = [], []
        with torch.no_grad():
            for spec1024, spec512, label in tqdm(recomp_test_loader, desc="Robustness Testing"):
                spec1024, spec512, label = spec1024.to(device), spec512.to(device), label.to(device)
                logits, _ = model(spec1024, spec512)
                probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                preds = (probs >= val_selected_threshold).astype(int)
                recomp_labels.extend(label.cpu().numpy())
                recomp_preds.extend(preds)

        recomp_f1 = f1_score(recomp_labels, recomp_preds, zero_division=0)
        robustness["recomp_test_f1"] = recomp_f1
        if not math.isnan(test_metrics["test_f1"]):
            robustness["delta_f1"] = test_metrics["test_f1"] - recomp_f1

    total_train_time = time.time() - start_time
    avg_epoch_time = total_train_time / len(history["train_loss"]) if history["train_loss"] else 0

    # ===== PREPARE RESULT ROW =====
    result_row = {
        "variant": variant_name,
        "seed": seed,
        "best_epoch": best_epoch,
        "mean_auc": np.nanmean(history["val_auc"]),
        "std_auc": np.nanstd(history["val_auc"]),
        "early_stopped": bool(early_stopped),
        "val_selected_threshold": val_selected_threshold,
        "val_thr_f1": val_thr_f1,
        "mean_val_f1": np.mean(history["val_f1"]) if len(history["val_f1"])>0 else np.nan,
        "best_val_f1": best_val_f1,
        "best_val_auc": best_val_auc,
        "test_f1": test_metrics["test_f1"],
        "test_acc": test_metrics["test_acc"],
        "test_prec": test_metrics["test_prec"],
        "test_rec": test_metrics["test_rec"],
        "recomp_test_f1": robustness["recomp_test_f1"],
        "delta_f1_recomp": robustness["delta_f1"],
        "ER_bin": er_bin,
        "optimizer": optimizer_name,
        "lr": lr,
        "batch_size": batch_size,
        "patience": patience,
        "final_train_loss": history["train_loss"][-1] if history["train_loss"] else np.nan,
        "early_stopped_epoch": best_epoch if early_stopped else np.nan,
        "total_epochs_run": len(history["train_loss"]),
        "n_params": n_params,
        "model_size_MB": sum(p.numel() * p.element_size() for p in model.parameters()) / (1024**2),
        "train_time_sec": total_train_time,
        "avg_epoch_time_sec": avg_epoch_time,
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda if torch.cuda.is_available() else "cpu",
        "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor(),
    }

    # Append to CSV
    df = pd.DataFrame([result_row])
    if not os.path.exists(results_file):
        df.to_csv(results_file, index=False)
    else:
        df.to_csv(results_file, mode="a", header=False, index=False)

    return result_row, history


# =====================================================
# DRIVER FUNCTION
# =====================================================

def run_top_variants(variant_models, seeds, train_loader, val_loader, test_loader=None, recomp_test_loader=None):
    """
    Run multiple model variants with multiple seeds
    """
    print("\n Verifying data loaders...")
    if not verify_data_loader(train_loader, "Train"):
        raise ValueError(" Train loader verification failed! Fix your data pipeline before training.")
    if not verify_data_loader(val_loader, "Validation"):
        raise ValueError(" Validation loader verification failed!")

    run_rows = []
    for variant in variant_models:
        vname = variant["name"]
        constructor = variant["constructor"]
        er_bin = variant.get("ER_bin", "NA")
        per_variant_rows = []
        print(f"\n{'='*80}")
        print(f"=== Running variant {vname} (ER_bin={er_bin}) ===")
        print(f"{'='*80}")

        for seed in seeds:
            # Check if already completed
            if is_seed_completed(seed, vname, results_file):
                print(f"  [{vname}][Seed {seed}] Already completed - skipping")
                try:
                    df = pd.read_csv(results_file)
                    existing_row = df[(df["variant"] == vname) & (df["seed"] == seed)].iloc[-1].to_dict()
                    per_variant_rows.append(existing_row)
                except:
                    pass
                continue

            print(f"\n [{vname}][Seed {seed}] Starting/resuming training")
            row, history = train_and_validate_variant(
                seed, vname, constructor, train_loader, val_loader,
                test_loader=test_loader, er_bin=er_bin, recomp_test_loader=recomp_test_loader
            )
            per_variant_rows.append(row)
            run_rows.append(row)

        # Compute aggregated statistics
        best_val_f1s = [r["best_val_f1"] for r in per_variant_rows if r["best_val_f1"] is not None and not math.isnan(r["best_val_f1"])]
        best_val_aucs = [r["best_val_auc"] for r in per_variant_rows if r["best_val_auc"] is not None and not math.isnan(r["best_val_auc"])]
        test_f1s = [r["test_f1"] for r in per_variant_rows if r["test_f1"] is not None and not math.isnan(r["test_f1"])]

        mean_f1, std_f1, ci_f1 = mean_std_ci(best_val_f1s)
        mean_auc, std_auc, ci_auc = mean_std_ci(best_val_aucs)
        mean_test_f1, std_test_f1, ci_test_f1 = mean_std_ci(test_f1s)

        summary_row = {
            "variant": vname,
            "seed": "aggregate",
            "n_seeds": len(per_variant_rows),
            "mean_best_val_f1": mean_f1,
            "std_best_val_f1": std_f1,
            "ci95_best_val_f1": ci_f1,
            "mean_best_val_auc": mean_auc,
            "std_best_val_auc": std_auc,
            "ci95_best_val_auc": ci_auc,
            "mean_test_f1": mean_test_f1,
            "std_test_f1": std_test_f1,
            "ci95_test_f1": ci_test_f1,
            "ER_bin": er_bin
        }

        # Save aggregate summary
        sdf = pd.DataFrame([summary_row])
        agg_file = os.path.join(checkpoint_dir, f"{vname}_aggregate_summary.csv")
        sdf.to_csv(agg_file, index=False)

        print(f"\n{'='*80}")
        print(f"[{vname}] AGGREGATE RESULTS:")
        print(f"  Mean Val F1: {mean_f1:.4f} ± {std_f1:.4f} (95% CI ±{ci_f1:.4f})")
        print(f"  Mean Val AUC: {mean_auc:.4f} ± {std_auc:.4f}")
        print(f"  Mean Test F1: {mean_test_f1:.4f} ± {std_test_f1:.4f}")
        print(f"  Saved to: {agg_file}")
        print(f"{'='*80}\n")

        # ER-bin aggregation
        if os.path.exists(results_file):
            try:
                df_all = pd.read_csv(results_file)
                er_group = df_all[df_all["variant"] == vname].groupby("ER_bin").agg(
                    mean_best_val_f1=pd.NamedAgg(column="best_val_f1", aggfunc=lambda x: np.nanmean(x)),
                    std_best_val_f1=pd.NamedAgg(column="best_val_f1", aggfunc=lambda x: np.nanstd(x)),
                    n_runs=pd.NamedAgg(column="seed", aggfunc="count")
                ).reset_index()
                er_group_file = os.path.join(checkpoint_dir, f"{vname}_ER_bin_summary.csv")
                er_group.to_csv(er_group_file, index=False)
                print(f"[{vname}] ER-bin aggregation saved to {er_group_file}")
            except Exception as e:
                print(f"Could not compute ER-bin aggregation: {e}")

    return run_rows


# =====================================================
# EXAMPLE USAGE
# =====================================================

if __name__ == "__main__":
    # Define model variant(s)
    variant_models = [
        {
            "name": "ExplainableSteganalysis",
            "constructor": lambda: ImprovedExplainableSteganalysisModel(),
            "ER_bin": "NA"
        }
    ]

    # Seeds to use
    seeds = [0, 42, 1337, 2025, 9999]

    results = run_top_variants(
        variant_models,
        seeds,
        train_loader,
        val_loader,
        test_loader=test_loader,
        recomp_test_loader=recomp_test_loader if 'recomp_test_loader' in globals() else None
    )

    print("\n" + "="*80)
    print(" ALL TRAINING COMPLETED!")
    print("="*80)
    print(f"Results saved to: {results_file}")
    print(f"Per-seed histories saved to: {per_seed_history_dir}")
    print("="*80)